# 1. Grafana Logs Data Exploration

This notebook explores the structure and characteristics of the Grafana logs generated for MOZAIC.

## Objectives
1. Load and parse Grafana logs
2. Understand the data structure
3. Analyze temporal patterns
4. Examine service distribution
5. Identify metric types and dashboards

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully")

## 1. Load Grafana Logs

In [ ]:
def load_grafana_logs(file_path, sample_size=None):
    """
    Load Grafana logs from JSONL file.
    
    Args:
        file_path: Path to the JSONL file
        sample_size: Number of logs to sample (None for all)
    
    Returns:
        list of dictionaries
    """
    logs = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if sample_size and i >= sample_size:
                break
            try:
                logs.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                continue
    return logs

# Load logs (using full dataset)
log_file = '../output/grafana/logs_2024-01-01.jsonl'
print(f"Loading logs from {log_file}...")
logs = load_grafana_logs(log_file)

print(f"\n✅ Loaded {len(logs):,} log entries")
print(f"Sample log entry:")
print(json.dumps(logs[0], indent=2))

## 2. Convert to DataFrame for Analysis

In [ ]:
def logs_to_dataframe(logs):
    """
    Convert logs to a structured DataFrame.
    """
    records = []
    for log in logs:
        record = {
            'dashboard': log.get('dashboard'),
            'panel_id': log.get('panel', {}).get('id'),
            'panel_title': log.get('panel', {}).get('title'),
            'panel_type': log.get('panel', {}).get('type'),
            'datasource': log.get('panel', {}).get('datasource'),
            'query_expr': log.get('target', {}).get('expr'),
            'legend_format': log.get('target', {}).get('legendFormat'),
            'service': log.get('tags', {}).get('service'),
            'environment': log.get('tags', {}).get('environment'),
            'cluster': log.get('tags', {}).get('cluster'),
            'executed_query': log.get('meta', {}).get('executedQueryString'),
            'viz_type': log.get('meta', {}).get('preferredVisualisationType')
        }
        
        # Extract datapoint info
        datapoints = log.get('datapoints', [[None, None]])
        if datapoints:
            record['value'] = datapoints[0][0]
            record['timestamp'] = datapoints[0][1]
        
        records.append(record)
    
    df = pd.DataFrame(records)
    
    # Convert timestamp from milliseconds to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    
    return df

df = logs_to_dataframe(logs)
print(f"\n✅ Created DataFrame with shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 3. Basic Statistics

In [ ]:
print("=" * 80)
print("GRAFANA LOGS OVERVIEW")
print("=" * 80)

print(f"\n📊 Total Entries: {len(df):,}")
print(f"📅 Time Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"⏱️  Duration: {(df['timestamp'].max() - df['timestamp'].min())}")

print(f"\n🎯 Unique Dashboards: {df['dashboard'].nunique()}")
print(f"📊 Unique Panels: {df['panel_id'].nunique()}")
print(f"🔧 Unique Services: {df['service'].nunique()}")
print(f"🌍 Unique Clusters: {df['cluster'].nunique()}")

print("\n" + "=" * 80)

# Data quality
print("\n📋 Data Quality:")
print(df.isnull().sum())

## 4. Dashboard Distribution

In [ ]:
# Dashboard distribution
dashboard_counts = df['dashboard'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
dashboard_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Dashboard Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Dashboard', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
dashboard_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Dashboard Percentage', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n📊 Top 10 Dashboards:")
print(dashboard_counts.head(10))

## 5. Service Distribution

In [ ]:
# Service distribution
service_counts = df['service'].value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
service_counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Service Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Service', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n🔧 Service Statistics:")
print(service_counts)

## 6. Panel Type Distribution

In [ ]:
# Panel type distribution
panel_type_counts = df['panel_type'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
panel_type_counts.plot(kind='bar', ax=ax, color='mediumseagreen')
ax.set_title('Panel Type Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Panel Type', fontsize=12)
ax.set_ylabel('Count', fontsize=12)

plt.tight_layout()
plt.show()

print("\n📊 Panel Types:")
print(panel_type_counts)

## 7. Temporal Analysis

In [ ]:
# Temporal patterns
hourly_counts = df.groupby('hour').size()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Hourly distribution
axes[0].plot(hourly_counts.index, hourly_counts.values, marker='o', linewidth=2, color='steelblue')
axes[0].fill_between(hourly_counts.index, hourly_counts.values, alpha=0.3)
axes[0].set_title('Hourly Log Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour of Day', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Timeline view
timeline = df.groupby(df['timestamp'].dt.floor('10T')).size()
axes[1].plot(timeline.index, timeline.values, linewidth=1.5, color='coral')
axes[1].set_title('Timeline View (10-minute intervals)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Metric Value Distribution

In [ ]:
# Value distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram
axes[0].hist(df['value'].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Metric Value Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Value', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_yscale('log')

# Box plot
axes[1].boxplot(df['value'].dropna(), vert=True)
axes[1].set_title('Metric Value Box Plot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Value', fontsize=12)

plt.tight_layout()
plt.show()

print("\n📊 Value Statistics:")
print(df['value'].describe())

## 9. Query Pattern Analysis

In [ ]:
# Analyze query patterns
query_patterns = df['query_expr'].value_counts().head(20)

print("\n🔍 Top 20 Query Patterns:")
for i, (query, count) in enumerate(query_patterns.items(), 1):
    print(f"{i:2d}. [{count:5,}x] {query[:100]}..." if len(query) > 100 else f"{i:2d}. [{count:5,}x] {query}")

## 10. Panel Title Analysis

In [ ]:
# Panel title distribution
panel_titles = df['panel_title'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 8))
panel_titles.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_title('Top 15 Panel Titles', fontsize=14, fontweight='bold')
ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Panel Title', fontsize=12)

plt.tight_layout()
plt.show()

## 11. Dashboard-Service Matrix

In [ ]:
# Create dashboard-service matrix
dashboard_service_matrix = df.groupby(['dashboard', 'service']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(dashboard_service_matrix, cmap='YlOrRd', annot=True, fmt='d', ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Dashboard-Service Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Service', fontsize=12)
ax.set_ylabel('Dashboard', fontsize=12)

plt.tight_layout()
plt.show()

## 12. Key Insights Summary

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

print(f"\n1. Dataset Size: {len(df):,} log entries")
print(f"2. Time Coverage: {df['timestamp'].max() - df['timestamp'].min()}")
print(f"3. Services Monitored: {df['service'].nunique()} unique services")
print(f"4. Dashboards: {df['dashboard'].nunique()} unique dashboards")
print(f"5. Panel Types: {', '.join(df['panel_type'].unique())}")
print(f"6. Most Active Service: {df['service'].value_counts().index[0]} ({df['service'].value_counts().values[0]:,} entries)")
print(f"7. Most Used Dashboard: {df['dashboard'].value_counts().index[0]} ({df['dashboard'].value_counts().values[0]:,} entries)")
print(f"8. Peak Hour: {hourly_counts.idxmax()}:00 ({hourly_counts.max():,} entries)")
print(f"9. Avg Entries per Hour: {len(df) / 24:.0f}")
print(f"10. Metric Value Range: [{df['value'].min():.2f}, {df['value'].max():.2f}]")

print("\n" + "="*80)
print("✅ Exploration Complete - Ready for Feature Engineering")
print("="*80)

## 13. Export Summary Statistics

In [ ]:
# Export summary for next notebooks
summary = {
    'total_entries': len(df),
    'num_services': df['service'].nunique(),
    'num_dashboards': df['dashboard'].nunique(),
    'num_panels': df['panel_id'].nunique(),
    'time_range': str(df['timestamp'].max() - df['timestamp'].min()),
    'services': df['service'].unique().tolist(),
    'dashboards': df['dashboard'].unique().tolist()
}

with open('../output/grafana/exploration_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Summary exported to: output/grafana/exploration_summary.json")